# Endurance Pharmacology Explorer

Predicting how endurance drugs bind their protein targets using **Chai-1**.

Structures are colored by **pLDDT confidence**: dark blue = high confidence (>90), light blue = confident (70-90), yellow = uncertain (50-70), orange/red = low confidence (<50). When the binding pocket glows blue, the model is sure. When it's orange, it's guessing.

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from pathlib import Path
import numpy as np
import py3Dmol

RESULTS = Path('../results')

def load_scores(pair_name, model_idx):
    s = np.load(RESULTS / pair_name / f'scores.model_idx_{model_idx}.npz')
    return {'agg': float(s['aggregate_score'][0]), 'ptm': float(s['ptm'][0]), 'iptm': float(s['iptm'][0])}

def render(pair_name, model_idx, width=900, height=550):
    """Render structure colored by pLDDT. Blue=confident, orange=uncertain."""
    cif = (RESULTS / pair_name / f'pred.model_idx_{model_idx}.cif').read_text()
    scores = load_scores(pair_name, model_idx)
    
    view = py3Dmol.view(width=width, height=height)
    view.addModel(cif, 'cif')
    
    # Protein: cartoon colored by pLDDT (B-factor)
    view.setStyle({'and': [{'not': {'resn': 'LIG2'}}, {'not': {'hetflag': True}}]},
                  {'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 30, 'max': 100}}})
    
    # Ligand/drug: bright sticks
    view.setStyle({'or': [{'resn': 'LIG2'}, {'hetflag': True}]},
                  {'stick': {'radius': 0.25, 'colorscheme': 'whiteCarbon'}})
    
    # Transparent surface around ligand binding region
    view.addSurface(py3Dmol.VDW, {'opacity': 0.15, 'color': 'white'},
                    {'within': {'distance': 6, 'sel': {'or': [{'resn': 'LIG2'}, {'hetflag': True}]}}})
    
    view.zoomTo()
    
    print(f'  pTM: {scores["ptm"]:.3f} | ipTM: {scores["iptm"]:.3f} | Aggregate: {scores["agg"]:.3f}')
    return view

def render_peptide(pair_name, model_idx, width=900, height=550):
    """Render peptide-protein interaction. Target=cartoon, peptide=sticks+cartoon."""
    cif = (RESULTS / pair_name / f'pred.model_idx_{model_idx}.cif').read_text()
    scores = load_scores(pair_name, model_idx)
    
    view = py3Dmol.view(width=width, height=height)
    view.addModel(cif, 'cif')
    
    # Chain A (target protein): cartoon by pLDDT
    view.setStyle({'chain': 'A'},
                  {'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 30, 'max': 100}}})
    
    # Chain B (peptide drug): cartoon + sticks, hot pink
    view.setStyle({'chain': 'B'},
                  {'cartoon': {'color': '#FF1493', 'opacity': 0.9},
                   'stick': {'radius': 0.2, 'color': '#FF1493'}})
    
    view.zoomTo()
    
    print(f'  pTM: {scores["ptm"]:.3f} | ipTM: {scores["iptm"]:.3f} | Aggregate: {scores["agg"]:.3f}')
    return view

print('Ready.')

Ready.


---
## Results Overview

| Target | Category | pTM | ipTM | Protein pLDDT | Ligand pLDDT | Verdict |
|--------|----------|-----|------|---------------|-------------|--------|
| **ITPP → Hemoglobin** | Allosteric | 0.889 | **0.849** | 92.0 | 44.7 | Excellent |
| **Cardarine → PPARδ** | Lock-and-key | 0.734 | **0.744** | 74.8 | 88.5 | Very good |
| **Roxadustat → PHD2** | Enzyme inhibitor | 0.628 | **0.464** | 68.3 | 72.7 | Solid |
| **TB-500 → Actin** | Peptide | 0.921 | 0.172 | 83.7 | 31.9 | Fold great, binding uncertain |
| **Ivabradine → HCN4** | Ion channel | 0.765 | 0.128 | 86.7 | 18.3 | Fold great, binding uncertain |
| **BPC-157 → VEGFR2** | Peptide (novel) | 0.490 | 0.114 | 79.9 | 39.9 | Frontier prediction |

---
## Tier 1: The Triumphs

### ITPP + Hemoglobin (ipTM: 0.849)

ITPP binds allosterically to the hemoglobin tetramer — outside the heme pocket — stabilizing the T-state to right-shift the O2 dissociation curve. More oxygen dumped into tissues per red cell pass.

Chai-1 nails this: the tetramer folds beautifully (pTM 0.89, pLDDT 92) and places the ligand with the highest binding confidence of any target. Note the deep blue protein — the model is certain about the fold.

In [3]:
render('itpp_hemoglobin', 1).show()

  pTM: 0.889 | ipTM: 0.849 | Aggregate: 0.857


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Cardarine (GW501516) + PPARδ (ipTM: 0.744)

PPARδ agonist. Upregulates fatty acid oxidation, shifts muscle fibers toward type I (slow-twitch), drives mitochondrial biogenesis via PGC-1α. The endurance compound that made mice run 70% longer.

Ligand pLDDT of **88.5** — the highest of any drug in the roster. The model places Cardarine deep in the PPARδ ligand-binding domain with high confidence. Look at the blue pocket hugging the white sticks.

In [4]:
render('cardarine_ppard', 2).show()

  pTM: 0.734 | ipTM: 0.744 | Aggregate: 0.742


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## Tier 2: Solid

### Roxadustat (FG-4592) + PHD2 (ipTM: 0.464)

Competitive inhibitor of prolyl hydroxylase PHD2. Blocks the oxygen sensor that degrades HIF-1α, mimicking hypoxia at sea level. Stabilized HIF-1α drives EPO transcription and VEGF expression. FDA-approved for anemia — oral altitude training in a pill.

Ligand pLDDT 72.7 with the **lowest spread across models** (16.9Å) — most consistent binding pose of any target. The model agrees on where this drug goes.

In [5]:
render('roxadustat_phd2', 4).show()

  pTM: 0.628 | ipTM: 0.464 | Aggregate: 0.497


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## Tier 3: The Frontier

These are the predictions where Chai-1 is uncertain. This is the interesting part.

### Ivabradine + HCN4 (ipTM: 0.128)

Open-channel blocker of the HCN4 funny current. Ivabradine must enter the **open pore** of a transmembrane ion channel — a deep, narrow, dynamic pocket. The protein fold is excellent (pTM 0.77, pLDDT 86.7) but the ligand placement is essentially random across the 5 models (42Å spread).

**Why it fails:** Ion channel pores are conformationally dynamic. The binding site only exists when the channel is open. Static structure prediction can't capture this.

In [6]:
render('ivabradine_hcn4', 3).show()

  pTM: 0.765 | ipTM: 0.128 | Aggregate: 0.255


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### TB-500 + G-Actin (ipTM: 0.172)

Active heptapeptide of Thymosin β4. Sequesters G-actin monomers to buffer cytoskeletal remodeling. The actin fold is outstanding (pTM **0.921**, highest of any target) but the 7-residue peptide placement is uncertain (52Å spread).

**Why it struggles:** A 7-residue peptide has too many degrees of freedom and too few intramolecular contacts for the model to anchor it confidently. The interaction surface is also shallow.

In [7]:
render_peptide('tb500_actin', 0).show()

  pTM: 0.921 | ipTM: 0.172 | Aggregate: 0.322


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### BPC-157 + VEGFR2 (ipTM: 0.114)

15-amino-acid gastric peptide docking into the massive VEGFR2 extracellular domain (~745 residues). No known co-crystal structure exists — this is a genuinely novel prediction.

**Why it's the hardest:** Largest target protein + flexible peptide + no structural prior = maximum uncertainty. The 62.7Å spread across models confirms this. But the prediction still generates physically reasonable contacts — it's just not converged.

This is exactly the kind of frontier problem Chai Discovery is working to solve.

In [8]:
render_peptide('bpc157_vegfr2', 1).show()

  pTM: 0.490 | ipTM: 0.114 | Aggregate: 0.189


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## What This Tells Us About Chai-1

| Chai-1 excels at | Chai-1 struggles with |
|------|------|
| Protein fold prediction (pTM 0.63–0.92 across all 6) | Small molecule placement in dynamic pockets (ion channels) |
| Small molecule docking into defined binding pockets (PPARδ, PHD2) | Short flexible peptide docking (7-15 residues) |
| Multi-chain complex assembly (hemoglobin tetramer) | Large receptor + peptide combinations (VEGFR2) |
| Allosteric binding when the pocket is structurally defined (ITPP-Hb) | Conformationally dependent binding sites |